# QC for final assemblies

1.) Input QC
- PoreC
- ReadLengths UL
- ReadLengths HQ_herro

2.) Assembly QC

In [17]:
library(tidyverse)
library(ggplot2)

In [18]:
samples <- c(
    "GE-MED-T2T00",
    "GE-MED-T2T04"
)

## 1.) Input QC

In [37]:
source("../scripts/02_plot_read_stats.R")

types <- c("HQ_herro.50x", "UL.70x")

dt_input_qc <- expand.grid(sample = samples, type = types) %>%
    mutate(path = paste0("../../assembly/input_qc/", sample, "/", sample, ".", type, "/read_stats.txt"))

dt_input_qc

sample,type,path
<fct>,<fct>,<chr>
GE-MED-T2T00,HQ_herro.50x,../../assembly/input_qc/GE-MED-T2T00/GE-MED-T2T00.HQ_herro.50x/read_stats.txt
GE-MED-T2T04,HQ_herro.50x,../../assembly/input_qc/GE-MED-T2T04/GE-MED-T2T04.HQ_herro.50x/read_stats.txt
GE-MED-T2T00,UL.70x,../../assembly/input_qc/GE-MED-T2T00/GE-MED-T2T00.UL.70x/read_stats.txt
GE-MED-T2T04,UL.70x,../../assembly/input_qc/GE-MED-T2T04/GE-MED-T2T04.UL.70x/read_stats.txt


In [38]:
dt_l <- list()
for (i in 1:nrow(dt_input_qc)) {
    print(paste("Processing", dt_input_qc$sample[i]))
    dt_l[[dt_input_qc$path[i]]] <- process_sequencing_file(dt_input_qc$path[i])
} 

[1] "Processing GE-MED-T2T00"
[1] "Reading file: ../../assembly/input_qc/GE-MED-T2T00/GE-MED-T2T00.HQ_herro.50x/read_stats.txt"
[1] "Calculating summary statistics"
[1] "Calculating 1D density for read length"
[1] "Calculating 1D density for mean quality"
[1] "Calculating 2D density"


Warning message in calculate_2d_density(result$plot_data):
“Skipping 2D density for sample GE-MED-T2T00_HQ_herro.50x due to insufficient variation”


[1] "Processing GE-MED-T2T04"
[1] "Reading file: ../../assembly/input_qc/GE-MED-T2T04/GE-MED-T2T04.HQ_herro.50x/read_stats.txt"
[1] "Calculating summary statistics"
[1] "Calculating 1D density for read length"
[1] "Calculating 1D density for mean quality"
[1] "Calculating 2D density"


Warning message in calculate_2d_density(result$plot_data):
“Skipping 2D density for sample GE-MED-T2T04_HQ_herro.50x due to insufficient variation”


[1] "Processing GE-MED-T2T00"
[1] "Reading file: ../../assembly/input_qc/GE-MED-T2T00/GE-MED-T2T00.UL.70x/read_stats.txt"
[1] "Calculating summary statistics"
[1] "Calculating 1D density for read length"
[1] "Calculating 1D density for mean quality"
[1] "Calculating 2D density"


Warning message in bkde2D(cbind(x_vals, y_vals), bandwidth = c(h1, h2), gridsize = c(n, :
“Binning grid too coarse for current (small) bandwidth: consider increasing 'gridsize'”


[1] "Processing GE-MED-T2T04"
[1] "Reading file: ../../assembly/input_qc/GE-MED-T2T04/GE-MED-T2T04.UL.70x/read_stats.txt"
[1] "Calculating summary statistics"
[1] "Calculating 1D density for read length"
[1] "Calculating 1D density for mean quality"
[1] "Calculating 2D density"


Warning message in bkde2D(cbind(x_vals, y_vals), bandwidth = c(h1, h2), gridsize = c(n, :
“Binning grid too coarse for current (small) bandwidth: consider increasing 'gridsize'”


In [55]:
for (i in 1:length(dt_l)) {
    dt_l[[i]]$summary_stats$path <- names(dt_l)[i]
    dt_l[[i]]$read_length_density$path <- names(dt_l)[i]
    dt_l[[i]]$quality_density$path <- names(dt_l)[i]
    dt_l[[i]]$density_2d$path <- names(dt_l)[i]
}
dt_stats <- bind_rows(lapply(dt_l, function(x) x$summary_stats)) %>%
    inner_join(dt_input_qc, by = c("path" = "path")) %>%
    select(c(-sample_name))
dt_read_length_density <- bind_rows(lapply(dt_l, function(x) x$read_length_density)) %>%
        inner_join(dt_input_qc, by = c("path" = "path"))%>%
    select(c(-sample_name))
dt_quality_density <- bind_rows(lapply(dt_l, function(x) x$quality_density))%>%
        inner_join(dt_input_qc, by = c("path" = "path"))%>%
    select(c(-sample_name))
dt_density_2d <- bind_rows(lapply(dt_l, function(x) x$density_2d))%>%
        inner_join(dt_input_qc, by = c("path" = "path"))%>%
    select(c(-sample_name))


In [56]:
dt_stats

N50,mean_quality,median_quality,total_bases,num_reads,mean_read_length,median_read_length,yield_above_80kb,yield_above_100kb,yield_above_200kb,yield_above_500kb,reads_above_1MB,path,sample,type
<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<fct>,<fct>
80466,33.00000,33.00000,160000022814,2066094,77440.82,64294,80618140989,56517647266,9702465035,30196590,0,../../assembly/input_qc/GE-MED-T2T00/GE-MED-T2T00.HQ_herro.50x/read_stats.txt,GE-MED-T2T00,HQ_herro.50x
110550,33.00000,33.00000,160000007054,1495219,107007.74,89944,120953482092,92183929678,23968796860,658038978,0,../../assembly/input_qc/GE-MED-T2T04/GE-MED-T2T04.HQ_herro.50x/read_stats.txt,GE-MED-T2T04,HQ_herro.50x
122012,24.98229,24.98229,89135967621,720989,123630.14,108256,89134847621,63954213335,12712024242,537756923,75,../../assembly/input_qc/GE-MED-T2T00/GE-MED-T2T00.UL.70x/read_stats.txt,GE-MED-T2T00,UL.70x
137001,24.77841,24.77841,137813393040,1014245,135877.81,115656,137811793040,108199024306,32430409794,1509263333,67,../../assembly/input_qc/GE-MED-T2T04/GE-MED-T2T04.UL.70x/read_stats.txt,GE-MED-T2T04,UL.70x


### PoreC QC, using results from wf-porec pipeline

In [ ]:
#todo